# Monte Carlo error and maximization bias

**Learning goals:** verify the $1/\sqrt{N}$ error rate and reproduce maximization bias with independent simulations.

**Predict first:** if the number of equally good actions doubles, does the maximum estimator become more or less optimistic?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

rng = np.random.default_rng(2026)
sample_sizes = 2 ** np.arange(0, 11)
rmse = []
for n in sample_sizes:
    means = rng.normal(size=(4000, n)).mean(axis=1)
    rmse.append(np.sqrt(np.mean(means**2)))
rmse = np.asarray(rmse)

plt.loglog(sample_sizes, rmse, "o-", label="empirical")
plt.loglog(sample_sizes, 1 / np.sqrt(sample_sizes), "--", label="theory")
plt.xlabel("Samples N")
plt.ylabel("RMSE")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
@interact(actions=IntSlider(value=8, min=2, max=32, step=2))
def estimator_histogram(actions):
    local_rng = np.random.default_rng(7)
    first = local_rng.normal(size=(5000, actions))
    second = local_rng.normal(size=(5000, actions))
    selected = first.argmax(axis=1)
    maximum = first.max(axis=1)
    double = second[np.arange(len(second)), selected]
    plt.figure(figsize=(7, 3))
    plt.hist(maximum, bins=50, alpha=0.55, density=True, label="maximum")
    plt.hist(double, bins=50, alpha=0.55, density=True, label="double")
    plt.axvline(0, color="black", linestyle="--")
    plt.legend()
    plt.title(f"Biases: maximum={maximum.mean():.3f}, double={double.mean():.3f}")
    plt.show()

**Experiment:** give action 0 a true advantage of 0.5. When does double estimation become negatively biased because the noisy selector chooses the wrong action?

In [ ]:
fitted_slope = np.polyfit(np.log(sample_sizes), np.log(rmse), 1)[0]
assert -0.58 < fitted_slope < -0.42
assert rmse[-1] < rmse[0] / 20
print(f"Checks passed; fitted log-log slope = {fitted_slope:.3f}.")